# 54 — Results freeze: the last steps after notebooks 21, 22, 23, 24, 25

Run after the five figure notebooks. It (1) writes the PNG and CSV copies of `q6_half_of_cohort_heatmap` beside its PDF, (2) copies the tables the text cites that live only in `figures` into `Data_for_writing/csv/`, (3) completes and prunes `MANIFEST.md`, (4) builds `freeze/` (NUMBERS_I-V, csv/, Figures/, MANIFEST.md; CHANGES.md is kept).

In [ ]:
import sys, pathlib, shutil
PILLAR = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(PILLAR / "src"))
import matplotlib.pyplot as plt
from viz import config, thesis
from viz import thesis_latex_part2 as h2, figures_part5 as v5
from viz.thesis_latex_common import root
cfg = config.load()
ctx = thesis.context(cfg)
import thesis_style as ts; ts.use()                        # thesis base style, after the viz style
R = root(cfg)
print(R)

## 1. PNG and CSV of the heatmap beside its PDF

In [ ]:
n = 'q6_half_of_cohort_heatmap'
fig, data, chk = h2.draw(h2._item(n), ctx, h2.FIGS[n][0])
o = R / 'Figures/Part_II_design_space/II3_distribution_and_new_variety/auxiliary'
o.mkdir(parents=True, exist_ok=True)
fig.savefig(o / f'{n}.png', dpi=200)
data.to_csv(o / f'{n}.csv', index=False)
plt.close(fig)
print('checks (text over text, under 7 pt):', chk)

## 2. Tables the text cites that are written only to figures

In [ ]:
CITED_V2 = ['q2_largest_filer_per_class.csv']
for name in CITED_V2:
    src = next(v5.out_root(cfg).rglob(name))
    shutil.copy2(src, R / 'Data_for_writing' / 'csv' / name)
    print('copied', src.relative_to(v5.out_root(cfg)))

## 3. MANIFEST: add the files written outside the part exporters, drop lines whose file no longer exists

In [ ]:
IV2 = 'Figures/Part_IV_design_space_evolution/IV2_attributes_and_drivers/main/'
HEAT = 'Figures/Part_II_design_space/II3_distribution_and_new_variety/auxiliary/q6_half_of_cohort_heatmap'
EXTRA = [(f'{IV2}q6_iv2_merge_cvt_lc.{e}', 'yes', 'CVT and Lift + Cruise designs of 2012-15 and 2020-23 by tilting sets and propulsors per design; outlines and centre paths') for e in ('pdf', 'png', 'csv')] + \
        [(f'{IV2}q6_iv2_split_tp.{e}', 'yes', 'Tilt Propulsor splitting into one and two or more tilting sets, per cohort; three patent drawings') for e in ('pdf', 'png', 'csv')] + \
        [(f'{IV2}q6_iv2_merge_cvt_lc_distance.csv', 'yes', 'Gower distance (34 dimensions) of CVT* and of all CVT to Lift + Cruise per cohort, with the permutation band of the change 2012-15 to 2020-23')] + \
        [(f'{HEAT}.png', 'no (leaves the text)', 'the heatmap as PNG'), (f'{HEAT}.csv', 'no (leaves the text)', 'data behind q6_half_of_cohort_heatmap (figure folder copy)')] + \
        [('Data_for_writing/csv/q2_largest_filer_per_class.csv', 'yes', 'largest assignee per architecture, all years and in one cohort')]
mf = R / 'Data_for_writing' / 'MANIFEST.md'
lines = mf.read_text(encoding='utf-8').splitlines()
have = {l.split('`')[1] for l in lines if l.startswith('- `')}
lines += [f'- `latex/{p}` · {u} · {w}' for p, u, w in EXTRA if f'latex/{p}' not in have]
kept, dropped = [], []
for l in lines:
    if l.startswith('- `latex/'):
        p = R / l.split('`')[1].split('/', 1)[1]
        if not p.exists():
            dropped.append(l.split('`')[1]); continue
    kept.append(l)
seen, out = set(), []
for l in kept:
    if l not in seen:
        seen.add(l); out.append(l)
mf.write_text('\n'.join(out) + '\n', encoding='utf-8')
print(len(out), 'lines;', len(dropped), 'dropped (file gone):', dropped)

## 4. freeze/: the files the dissertation reads

In [ ]:
F = R.parent / 'freeze'
keep = (F / 'CHANGES.md').read_text(encoding='utf-8') if (F / 'CHANGES.md').exists() else None
checks = None
if (F / 'checks').exists():                        # the check sheets made by hand for the author are kept
    checks = R.parent / '_checks_keep'
    shutil.rmtree(checks, ignore_errors=True)
    shutil.copytree(F / 'checks', checks)
if F.exists():
    shutil.rmtree(F)
F.mkdir()
for n in ('NUMBERS_I.md', 'NUMBERS_II.md', 'NUMBERS_III.md', 'NUMBERS_IV.md', 'NUMBERS_V.md', 'MANIFEST.md'):
    shutil.copy2(R / 'Data_for_writing' / n, F / n)
shutil.copytree(R / 'Data_for_writing' / 'csv', F / 'csv')
shutil.copytree(R / 'Figures', F / 'Figures')
if checks is not None:
    shutil.copytree(checks, F / 'checks')
    shutil.rmtree(checks)
if keep is not None:
    (F / 'CHANGES.md').write_text(keep, encoding='utf-8')
files = sorted(p.relative_to(F) for p in F.rglob('*') if p.is_file())
print(len(files), 'files in', F)

## 5. The generated Results text of Q7 (Analysis III), with its numbers filled in
Written to `freeze/RESULTS_TEXT_Q7.md` from `viz/thesis_text_q7.py` and `viz.thesis_q7.numbers(ctx)`; nothing is typed.

In [ ]:
from viz import thesis_q7 as q7, thesis_text_q7 as tx7
n7 = q7.numbers(ctx)
parts = ['# Results text, Q7 (Analysis III, dominant design)\n', 'Generated from `viz/thesis_text_q7.py` with `viz.thesis_q7.numbers(ctx)`.\n',
         '## Method\n', tx7.METHOD.format(**n7), '']
parts += [f"## {r['sub']}. {r['title']}\n\n{r['text'].format(**n7)}\n" for r in tx7.RESULTS]
parts += ['## Answer\n', tx7.ANSWER.format(**n7), '', '## Appendix captions\n']
parts += [f"- `{k}`: {t.format(**n7)}" for k, t in tx7.APPENDIX.items()]
(F / 'RESULTS_TEXT_Q7.md').write_text('\n'.join(parts) + '\n', encoding='utf-8')
print('written', F / 'RESULTS_TEXT_Q7.md')

In [ ]:
# The corrected dataset (eVTOL_dataset_v1 or the public eVTOL_dataset): the aircraft ids written by notebooks 21-54 follow the dataset (<patent>_ad<N>).
# After notebooks 27-57, run again: python -m viz.dataset_ids <roots.output>
from viz import dataset_ids, dataset_public
if dataset_ids.is_v1(cfg.raw['roots']['labelled']) or dataset_public.is_public(cfg.raw['roots']['labelled']):
    print(dataset_ids.publish_ids(cfg.roots['output']), 'output files: ids renamed to _ad')